# Lab 2 standalone Colab run

Open only this notebook. You do not need to upload Python files, clone a repository or provide a result ZIP. The first cell verifies its embedded code and creates its internal helper files automatically. Required course inputs and public model weights download automatically. Use Runtime > Change runtime type > GPU; T4 is the course's nominal 16-GB option. The actual GPU identity is recorded, not assumed to be an RTX 4090.

Part 2 already completed on your RTX 4090. This notebook restores both original adapters and all 120 original images from your private Hugging Face repositories, together with their saved measured records. It then runs the corrected Parts 1 and 3. Original Part 2 timings are not presented as a new Colab training run. This also does not establish that Part 2 trained on a 16-GB card.

Before leaving it running, provide a Hugging Face read token for those private repositories in the hidden prompt, or use a Colab secret named `HF_TOKEN`. Never paste it into source code or this chat. Keep roughly 40 GB of disk available for a cold setup and at least 8 GiB available host RAM. Leave other GPU notebooks stopped. Colab can still disconnect or end a session; no code can guarantee against that.

Use Run All and wait for both real-model preflights to pass. Results resume only while the same runtime files remain available. At the end, download the combined ZIP and save this notebook with its outputs. Real human ratings, team records, permissions and final external submission actions remain separate.

In [ ]:
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, stat, subprocess, sys, zipfile

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
if not SMOKE_ONLY:
    try:
        visible_gpu = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=15)
    except (OSError, subprocess.SubprocessError) as error:
        raise RuntimeError("Select Runtime > Change runtime type > GPU, then reconnect and Run All. A CPU runtime cannot run this experiment.") from error
    print(visible_gpu.strip())

# All helper source is embedded below. No repository or .py upload is needed.
BUNDLE_SHA256 = '2b8b34323790255247a43f78d25e25b9c857ef59c5cdf24c808b55704404a839'
BUNDLE_BASE64 = (
    '...'
)

def unpack(data, expected_sha256, parent):
    """No pre-existing repo, git executable, or package import is required."""
    if hashlib.sha256(data).hexdigest() != expected_sha256:
        raise ValueError("The embedded notebook bundle is incomplete or changed.")
    root = Path(parent) / ("Lab2_Standalone_" + expected_sha256[:16])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if archive.testzip() is not None:
            raise ValueError("The embedded ZIP failed its integrity check.")
        manifest = json.loads(archive.read("STANDALONE_MANIFEST.json"))
        files = manifest["files"]
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Duplicate archive paths are not permitted.")
        if set(archive.namelist()) != set(files) | {"STANDALONE_MANIFEST.json"}:
            raise ValueError("The archive does not match its file manifest.")
        # Check every path and hash before creating or changing any file.
        for member in archive.infolist():
            name = member.filename
            path = PurePosixPath(name)
            if path.is_absolute() or ".." in path.parts or "\\" in name or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe embedded path.")
            if member.is_dir():
                raise ValueError("Only regular manifest files are permitted.")
            if name != "STANDALONE_MANIFEST.json" and hashlib.sha256(archive.read(name)).hexdigest() != files[name]:
                raise ValueError(f"Embedded source hash mismatch: {name}")
            target = root / name
            if target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
                raise ValueError("An extraction target escaped the standalone folder.")
            if target.is_file() and target.read_bytes() != archive.read(name):
                raise ValueError(f"Local edits at {target} were preserved. Use a new empty parent folder.")
        root.mkdir(parents=True, exist_ok=True)
        for name in archive.namelist():
            target = root / name
            if not target.is_file():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    return root

default_parent = Path("/content") if Path("/content").is_dir() else Path.cwd()
parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", str(default_parent))).expanduser().resolve()
ROOT = unpack(base64.b64decode(BUNDLE_BASE64), BUNDLE_SHA256, parent)
os.environ["LAB2_STANDALONE_ROOT"] = str(ROOT)
print("Embedded code verified:", BUNDLE_SHA256)
print("Source base commit:", json.loads((ROOT / "STANDALONE_MANIFEST.json").read_text())["source_commit"])

from pathlib import Path
import os, sys, subprocess, json, time

ROOT = Path(os.environ["LAB2_STANDALONE_ROOT"]).resolve()
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["LAB2_DEVICE"] = "cpu" if SMOKE_ONLY else "cuda"

# This override exists only for a developer CPU smoke, never for a GPU run.
test_python = os.environ.get("LAB2_STANDALONE_TEST_PYTHON", "") if SMOKE_ONLY else ""
if test_python:
    PYTHON = Path(test_python).expanduser().absolute()
    if not PYTHON.is_file():
        raise FileNotFoundError("The explicit CPU test interpreter does not exist.")
    subprocess.run([str(PYTHON), str(ROOT / "tools/preflight.py")], cwd=ROOT, check=True)
    (ROOT / ".lab2_runtime_cpu.json").write_text(json.dumps({"python": str(PYTHON), "cuda_requested": False}))
else:
    subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                    "--device", "cpu" if SMOKE_ONLY else "cuda"], cwd=ROOT, check=True)
    marker = ROOT / (".lab2_runtime_cpu.json" if SMOKE_ONLY else ".lab2_runtime_cuda.json")
    PYTHON = Path(json.loads(marker.read_text())["python"])
SEED = 266
print("Mode:", "CPU SMOKE ONLY: model calls are skipped" if SMOKE_ONLY else "FULL MODEL RUN")
print("Working folder:", ROOT)
print("Pinned interpreter:", PYTHON)

def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    command = [str(PYTHON), "-u", str(ROOT / "tools/monitored_run.py"), "--record", str(record),
               "--", str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"{Path(relative).name} stopped with exit {status}. Saved log: {log}. Run the final ZIP cell to retain all partial results.")


## Download and verify the course inputs

The PDFs, question CSV and knowledge-base manifest use a pinned source revision and checked SHA-256 values. Executable source is already embedded; this cell does not download Python scripts. Completed unchanged input files are kept when a download needs retrying.

In [ ]:
run_script("tools/standalone_colab.py", "resources")
DATA = ROOT / "part1_rag/data"
os.environ["LAB2_DATA_DIR"] = str(DATA)
os.environ.setdefault("QUESTIONS_PATH", str(DATA / "Questions.csv"))
print("Inputs ready:", DATA)

## Run the offline regression checks

These temporary fixtures check structure, data handling, checkpoints and export. They are not pretrained-model quality scores. The actual model preflights follow later.

In [ ]:
for name in ["test_lab_completion.py", "test_structured_responses.py", "test_constrained_decoding.py",
             "test_overnight_resume.py", "test_part2_submission.py", "test_result_download.py", "test_rtx4090_workflow.py"]:
    run_script("tools/" + name)

## Recover the original Part 2 without uploading any files

Use your own Hugging Face account's read token. It must be able to read both private adapters and the image dataset. Access is checked before expensive work. The original dataset split is restored as needed, then each saved adapter and generated image is checked against its recorded hash. Comparison panels are rebuilt from those same images, not generated again.

The measured values displayed in the saved Part 2 notebook are explicitly labeled as original RTX 4090 results. A completed restoration does not create new training evidence, human judgments, sharing rights or grader access.

In [ ]:
if SMOKE_ONLY:
    print("SKIPPED: private Hugging Face access, dataset download and original Part 2 restoration.")
    run_script("tools/standalone_colab.py", "record-part2")
else:
    temporary_token = False
    if not os.environ.get("HF_TOKEN"):
        try:
            from google.colab import userdata
            secret = userdata.get("HF_TOKEN")
        except Exception:
            secret = None
        if not secret:
            from getpass import getpass
            secret = getpass("Hugging Face READ token for your private Part 2 artifacts (hidden): ").strip()
        if not secret:
            raise ValueError("A read token is required. No private download, training or model evaluation was started.")
        os.environ["HF_TOKEN"] = secret
        del secret
        temporary_token = True
    try:
        run_script("tools/standalone_colab.py", "check-hub-access")
        run_script("part2_diffusion/src/prepare_data.py", "--output-dir", ROOT / "part2_diffusion/data/processed")
        run_script("tools/standalone_colab.py", "restore-part2")
        run_script("tools/standalone_colab.py", "record-part2")
        run_script("part3_agents/src/prepare_evaluation.py", "--download-adapter")
    finally:
        if temporary_token:
            os.environ.pop("HF_TOKEN", None)

## Check the real model paths on this GPU

The RAG model, image understanding, prompt rewriting, critic, image editor and domain adapter are exercised sequentially. Tiny diagnostic images are not final samples. Stay connected until this prints `BOTH REAL GPU PREFLIGHTS PASSED`. CPU smoke output is never accepted as a passed GPU preflight.

In [ ]:
if SMOKE_ONLY:
    print("SKIPPED: actual GPU model preflights. No Colab GPU readiness is certified.")
else:
    run_script("tools/overnight_preflight.py", "--part", "part1_rag", "--data-dir", DATA)
    run_script("tools/overnight_preflight.py", "--part", "part3_agents")
    print("BOTH REAL GPU PREFLIGHTS PASSED. The full sequential run starts next.")

## Run Part 1 and Part 3 and keep the executed notebooks

Part 1 produces the validated CSV and evaluates the dense baseline against the final system. Part 3 compares 15 scenarios with and without critique and ten safety cases for each. Each request keeps its own trace and image path. Models run sequentially; revisions are capped at two.

The embedded notebooks save after every cell under `output/executed`. Unchanged successful items resume after interruption while those files remain on the runtime. Errors stay visible and are not replaced by made-up successful results.

In [ ]:
run_script("tools/overnight_runner.py", *(["--smoke-only"] if SMOKE_ONLY else []))
print("Saved executed notebooks:", ROOT / "output/executed")

## Check the results and pending review

A successful script exit is not enough: saved model errors and disabled-model fixtures fail the computation check. The separate audit lists real ratings, reference review, team details and other submission requirements that still need attention. Reports remain drafts while those are pending.

In [ ]:
run_script("tools/standalone_colab.py", "complete", *(["--smoke-only"] if SMOKE_ONLY else []))
run_script("tools/prepare_rating_forms.py")
run_script("tools/final_submission_audit.py")
print("Computation record:", ROOT / "runtime_checks/standalone_completion.json")
print("The final cell below downloads your saved results; it does not submit them.")

## Download one ZIP with all available lab results

Run this cell after the overnight work. It bundles all available results for Parts 1, 2 and 3, executed notebooks, source/configuration, reports, adapters, generated images and diagnostic logs. Base-model downloads, environments, original datasets and old archives are excluded. Missing files are listed honestly in `BUNDLE_MANIFEST.json`.

The ZIP transfers through the Colab notebook connection, including a local kernel, without a localhost hyperlink. A native Windows/Mac runtime also copies it into its accessible Downloads folder. Browser location follows your download settings. Keep the kernel connected until transfer finishes. If the run fails, you can execute only this export cell to send the partial evidence; no GPU work is repeated.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, importlib.util

if "ROOT" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents,
        Path(os.environ.get("LAB2_ROOT", str(Path.home() / "DATA266/Lab-2"))),
        Path("/content/Lab2_Standalone_2b8b343237902552"), Path.cwd() / "Lab2_Standalone_2b8b343237902552", Path(os.environ.get("LAB2_STANDALONE_ROOT", "/content/Lab-2"))]
    ROOT = next((p.resolve() for p in candidates if (p / "tools/export_lab_results.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Set LAB2_ROOT to your existing Lab-2 folder. Do not rerun training just to export files.")

# This script only collects saved files. It does not install packages, train, or generate.
subprocess.run([sys.executable, str(ROOT / "tools/export_lab_results.py"), "--all"], cwd=ROOT, check=True)
bundle = json.loads((ROOT / "output/exports/latest_all_results.json").read_text())
print("Upload this ZIP in our chat:", bundle["archive"])
print("ZIP SHA256:", bundle["sha256"])
if globals().get("SMOKE_ONLY", False):
    print("CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.")
else:
    spec = importlib.util.spec_from_file_location("lab2_download_results", ROOT / "tools/download_results.py")
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloader.offer_download(bundle["archive"], auto_start=True)